# Introducción al Análisis de Datos con Python

# 3. Limpieza y de datos

En este notebook, realizaremos una serie de procesos de limpieza de datos que nos facilitaran el análisis y visualización. La limpieza de datos es el proceso de identificar, corregir o eliminar datos incorrectos, incompletos, irrelevantes o duplicados en un conjunto de datos. La limpieza de datos es importante para garantizar la precisión y la calidad de los datos y evitar errores en el análisis posterior. Algunas técnicas comunes de limpieza de datos incluyen la eliminación de valores atípicos, la eliminación de valores faltantes, la corrección de errores tipográficos y la eliminación de duplicados.

Continuaremos trabajando con las Estadísticas de Defunciones Registradas (EDR) de 2020, una base de datos generada por el Instituto Nacional de Estadística y Geografía (INEGI) de México.


## Estrategía de limpieza de datos

La **limpieza de datos** consiste en detectar y corregir errores, inconsistencias y valores atípicos. Los datos que se obtienen de fuentes del mundo real casi nunca están listos para ser utilizados directamente, ya que suelen contener errores de captura, formatos variados, o datos faltantes. Se estima que en un proyecto de datos el 80% del tiempo se dedica a limpiar y preparar los datos. Un análisis basado en datos sucios puede llevar a conclusiones incorrectas y a decisiones erróneas.

Primero, empezaremos planteando una pregunta, la cual orientará nuestro análisis. En este caso la pregunta es:  
> **¿Cuál fue el perfil demográfico de las las defunciones por COVID-19 en México durante el año 2020?**.

A partir de esta pregunta plantearemos la estrategia de limpieza:
1. **Selección de variables**: Determinar y seleccionar exclusivamente las variables y registros que son relevantes para el análisis. Esto incluye:
 * Seleccionar variables (columnas) relevantes.
 * Filtrar las defunciones que no ocurrieron en 2020. Esto nos ayudará a evitar datos atípicos que se registraron tardíamente.
 * Seleccionar las defunciones por COVID-19, las cuales tienen los códigos CIE-10: `'U071','U072'`.
4. **Manejo de valores faltantes y no especificados**: Los datos del INEGI no usan `NaN` para los valores ausentes, sino códigos como `9` o `9999` dependiendo de la variable. Sustituiremos estas codificaciones para que Python las reconozca como valores faltantes.
5. **Mapeo de catálogos**: Transformaremos los códigos numéricos de columnas como `sexo` a valores descriptivos más comprensibles como 'Hombre' o 'Mujer'.
6. **Manejo de columnas especiales**: Transformaremos la codificación de edad al INEGI  a años y crearemos una columna `datetime` con la fecha.
7. **Revisión y conversión de tipos de datos**: Nos aseguraremos de que cada columna tenga el tipo de dato correcto.
8. **Guardar el conjunto de datos limpio**: Finalmente, exportaremos el DataFrame, esto nos permitirá cargar directamente un conjunto de datos limpio en futuros análisis.

En primer lugar, cargaremos las bibliotecas y datos con los que se va a trabajar.

In [1]:
# Instalar e importar bibliotecas
import pandas as pd
import numpy as np

# Abrir datos
from google.colab import drive
drive.mount('/content/drive')

# ruta al folder de trabajo
folder_path = '/content/drive/MyDrive/Trabajo/Python_EDR2020/'

# ruta del archivo dentro del folder de trabajo
file_name = folder_path+'conjunto_de_datos_defunciones_registradas_2020_csv/conjunto_de_datos/conjunto_de_datos_defunciones_registrados_2020.csv'
df = pd.read_csv(file_name)
df


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,ent_regis,mun_regis,ent_resid,mun_resid,tloc_resid,loc_resid,ent_ocurr,mun_ocurr,tloc_ocurr,loc_ocurr,...,anio_cert,maternas,lengua,cond_act,par_agre,ent_ocules,mun_ocules,loc_ocules,razon_m,dis_re_oax
0,1,1,1,1,15,1,1,1,15,1,...,2020,NaN,2,9,88,88,888,8888,0,999
1,1,1,1,1,15,1,1,1,15,1,...,2020,NaN,9,9,88,88,888,8888,0,999
2,1,1,1,1,15,1,1,1,15,1,...,2020,NaN,2,2,88,88,888,8888,0,999
3,1,1,1,1,15,1,1,1,15,1,...,2020,NaN,9,9,88,88,888,8888,0,999
4,1,6,1,9,4,1,1,9,4,1,...,2020,NaN,2,1,88,88,888,8888,0,999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1086738,32,17,32,17,13,1,32,56,13,1,...,2020,NaN,2,2,88,88,888,8888,0,999
1086739,32,56,32,5,9,1,32,56,13,1,...,2020,NaN,8,8,88,88,888,8888,0,999
1086740,32,17,32,56,13,1,32,17,13,1,...,2020,NaN,2,1,88,88,888,8888,0,999
1086741,32,24,32,24,8,1,32,17,13,1,...,2020,NaN,2,2,88,88,888,8888,0,999


## Selección de datos

Primero, seleccionaremos las variables, o columnas, que son relevantes para el análisis. Dada nuestra pregunta de investigación revisaremos el diccionario de datos y seleccionaremos las variables que describen:
* Lugar: `'ent_ocurr', 'mun_ocurr'`
* Fecha: `'dia_ocurr', 'mes_ocurr', 'anio_ocur'`
* Causa de la defunción: `'causa_def'`
* Sexo y edad: `'sexo', 'edad', 'edad_agru'`

Para validar el proceso nos fijaremos en el tamaño del DataFrame y a final haremos una revisión.

In [2]:
# Seleccionar las columnas de interés
columnas = ['dia_ocurr', 'mes_ocurr', 'anio_ocur', 'ent_ocurr', 'mun_ocurr',
            'causa_def', 'sexo', 'edad', 'edad_agru']
df_covid = df[columnas]
# Imprimir tamaño resultante
print(f"Filas df filtrado: {df_covid.shape[0]}/{df.shape[0]}")
print(f"Colum df filtrado: {df_covid.shape[1]}/{df.shape[1]}")

Filas df filtrado: 1086743/1086743
Colum df filtrado: 9/59


Después, seleccionaremos las observaciones, o filas, que corresponden a defunciones ocurridas en el año 2020. Esto nos ayudará a evitar datos de años atípicos que se registraron tardíamente.

In [3]:
# Filtrar defunciones ocurridas en 2020
df_covid = df_covid[ df_covid['anio_ocur']==2020 ]
# Imprimir tamaño resultante
print(f"Filas df filtrado: {df_covid.shape[0]}/{df.shape[0]}")
print(f"Colum df filtrado: {df_covid.shape[1]}/{df.shape[1]}")

Filas df filtrado: 1069958/1086743
Colum df filtrado: 9/59


Para obtener una selección precisa de las defunciones por COVID-19, nos basaremos en una lista de códigos de la Clasificación Internacional de Enfermedades (CIE-10). Seleccionaremos dos los códigos asociados al COVID-19: `U071`y `U072`,

In [4]:
# Decesos por COVID-19
codigo_covid19 = df_covid['causa_def'].isin(['U071','U072'])
# Seleccionar defunciones por enfermedades respiratorias y COVID-19
df_covid = df_covid[codigo_covid19]
# Imprimir tamaño resultante
print(f"Filas df filtrado: {df_covid.shape[0]}/{df.shape[0]}")
print(f"Colum df filtrado: {df_covid.shape[1]}/{df.shape[1]}")

Filas df filtrado: 200263/1086743
Colum df filtrado: 9/59


Veamos el DataFrame resultante


In [5]:
df_covid

,dia_ocurr,mes_ocurr,anio_ocur,ent_ocurr,mun_ocurr,causa_def,sexo,edad,edad_agru
5,26,10,2020,1,1,U071,1,4058,16
19,26,10,2020,1,1,U071,1,4072,19
585,28,4,2020,1,1,U071,1,4064,17
627,13,5,2020,1,1,U071,1,4046,14
683,17,5,2020,1,1,U071,1,4045,14
...,...,...,...,...,...,...,...,...,...
1086737,31,12,2020,32,56,U071,2,4057,16
1086738,31,12,2020,32,56,U072,2,4068,18
1086740,11,12,2020,32,17,U071,1,4074,19
1086741,18,12,2020,32,17,U071,2,4081,21


## Manejo de valores faltantes y no especificados

El INEGI codifica los datos faltantes con valores específicos, como `999`, `9999`. Para que nuestro análisis sea preciso, modificaremos esta codificación para que estos valores se traten como `NaN`, lo que permite que las funciones de pandas los manejen adecuadamente.

Para entender cómo el INEGI codifica esta información, es necesario consultar los catálogos que acompañan a la base de datos. Por ejemplo:

  * En la columna **`sexo`**, el valor `9` se usa para indicar un sexo "No especificado".
  * En la columna **`dia_ocurr`**, el valor `99` significa "No especificado", mientras que `9` es un día válido.

Esta distinción es fundamental y subraya por qué la exploración de metadatos es una etapa tan importante.

Para nuestro caso, decidiremos sustituir los valores de no especificado solo en las columnas de fecha (`'dia_ocurr'`, `'mes_ocurr'`, `'anio_ocurr'`) por `NaN`, ya que estos códigos podrían causar un error de tipo o un análisis incorrecto al momento de generar la variable de fecha en formato `datetime`.

Para esto, usaremos la función `.replace()`, la cual busca un valor o conjunto de valores específicos en una Serie o un DataFrame y los sustituye por otro valor.

Sintaxis básica: `df['columna'].replace(valores_a_cambiar, nuevo_valor)`.

In [6]:
# Reemplazar valores no especificados por NaN
df_covid['dia_ocurr'] = df_covid['dia_ocurr'].replace(99, np.nan)
df_covid['mes_ocurr'] = df_covid['mes_ocurr'].replace(99, np.nan)
df_covid['anio_ocur'] = df_covid['anio_ocur'].replace(9999, np.nan)
# Verificar los cambios
cols = ['dia_ocurr', 'mes_ocurr', 'anio_ocur']
df_covid.loc[df_covid[cols].isna().any(axis=1), cols]

,dia_ocurr,mes_ocurr,anio_ocur
26243,NaN,NaN,2020
26340,NaN,NaN,2020
26859,NaN,4.0,2020
27496,NaN,NaN,2020
36297,NaN,5.0,2020
73301,NaN,NaN,2020
134498,NaN,NaN,2020
138035,NaN,NaN,2020
162550,NaN,NaN,2020
163959,NaN,NaN,2020


## Mapeo de catálogos


### Mapeo de catálogos con `.replace()`

Para que los datos sean más comprensibles por humanos, es recomendable convertir los códigos numéricos a valores descriptivos. Usaremos el método `.replace()` de pandas, el cual también funciona con diccionarios para mapear valores. Si un valor en el DataFrame no se encuentra en el diccionario, se mantendrá sin cambios.

Empezaremos con la columna `'sexo'`. Su catálogo es el siguiente:

  * `1`: Hombre
  * `2`: Mujer
  * `9`: No especificado

Con estos datos crearemos un diccionario, una estructura de datos con llaves y valores, que usaremos para mapear el catálogo.

Nota: también es posible hacer este proceso con la función `map()`.

In [7]:
# Definir el diccionario de mapeo para la columna sexo
mapeo_sexo = { 1:'Hombre', 2:'Mujer', 9:'No especificado' }
# Aplicar el reemplazo en la columna 'sexo'
df_covid['sexo'] = df_covid['sexo'].replace(mapeo_sexo)
# Mostrar los valores únicos para verificar el cambio
df_covid['sexo'].value_counts(dropna=False)

,count
sexo,
Hombre,128799
Mujer,71459
No especificado,5


En este momento, la columna `sexo` es de tipo `object`. SIn embargo, nos interesa que sea de tipo `categorical`, tanto por cuestiones de uso de memoria como para facilitar el análisis. Para convertir el tipo usaremos la función `.astype('category')`. Nota como cambia la descripción de la columna al final de la celda de output.



In [8]:
df_covid['sexo'] = df_covid['sexo'].astype('category')
df_covid['sexo']

,sexo
5,Hombre
19,Hombre
585,Hombre
627,Hombre
683,Hombre
...,...
1086737,Mujer
1086738,Mujer
1086740,Hombre
1086741,Mujer


La columna `edad_agru` clasifica las defunciones en rangos de edad predefinidos, lo cual es útil para análisis demográficos. Para hacer esta variable más legible, la mapearemos de códigos numéricos a descripciones textuales.

Esta columna es categórica **ordenada**, es decir, tiene una jerarquía interna. Podemos generar categorías ordenadas usando `CategoricalDtype`. Es necesario definir primero la estructura de la categoría y luego convertir la columna de interés a ese "tipo" de categoría.

Para revisar el mapeo y la conversión de tipo usaremos la función `.value_counts()` y ordenaremos la tabla de frecuencias resultante por el índice, el cual contiene los valores de la columna, con `.sort_index()`. Nota como el ordenamiento se hace de acuerdo al orden de la categoría definida, en lugar de ser alfabético.

In [9]:
from pandas.api.types import CategoricalDtype
# Construir el catálogo
mapeo_edad_agru = {
                    1:'Menores de un año', 2:'De un año', 3:'De 2', 4:'De 3', 5:'De 4', 6:'De 5 a 9',
                    7:'De 10 a 14', 8:'De 15 a 19', 9:'De 20 a 24', 10:'De 25 a 29', 11:'De 30 a 34',
                    12:'De 35 a 39', 13:'De 40 a 44', 14:'De 45 a 49', 15:'De 50 a 54', 16:'De 55 a 59',
                    17:'De 60 a 64', 18:'De 65 a 69', 19:'De 70 a 74', 20:'De 75 a 79', 21:'De 80 a 84',
                    22:'De 85 a 89', 23:'De 90 a 94', 24:'De 95 a 99', 25:'De 100 a 104', 26:'De 105 a 109',
                    27:'De 110 a 114', 28:'De 115 a 119', 29:'De 120 y más', 30:'No especificada'
                  }
# Aplicar el reemplazo en la columna 'edad_agru'
df_covid['edad_agru'] = df_covid['edad_agru'].replace(mapeo_edad_agru)
# Generar la categoría ordenada, usamos los valores del diccionario
cat_edad_agru = CategoricalDtype(categories=mapeo_edad_agru.values(), ordered=True)
# Convertir a la columna a la categoría ordenada
df_covid['edad_agru'] = df_covid['edad_agru'].astype(cat_edad_agru)
# Mostrar el conteo de valores únicos para verificar el cambio
df_covid['edad_agru'].value_counts().sort_index()

,count
edad_agru,
Menores de un año,209
De un año,54
De 2,24
De 3,20
De 4,13
De 5 a 9,61
De 10 a 14,89
De 15 a 19,248
De 20 a 24,677


### Mapeo de catálogos con `merge()`

En el EDR las causas de defunción se encuentras codificadas de acuerdo al  CIE-10, se encuentran en la carpeta de `catalogos` el archivo `causa_defuncion.CSV`. Si revisamos este archivo podemos ver que contiene 3965 causas, por lo que es complejo es escribir un diccionario de remplazo de manera manual. Para hacer este mapeo leeremos la tabla del catálogo desde el notebook y la guardaremos en una variable nueva para poder trabajar con ella. En este caso, dando la gran cantidad de valores, dejaremos la columna con el tipo `object`.

In [10]:
file_path = folder_path+'conjunto_de_datos_defunciones_registradas_2020_csv/catalogos/causa_defuncion.CSV'
mapeo_causa = pd.read_csv(file_path)
mapeo_causa.tail()

,cve,descrip
3961,Y850,Secuelas de accidente de vehículo de motor
3962,Y86X,Secuelas de otros accidentes
3963,Y881,Secuelas de incidentes ocurridos al paciente d...
3964,Y883,Secuelas de procedimientos médicos y quirúrgic...
3965,Y899,Secuelas de causa externa no especificada


El objetivo es usar este catálogo para añadir los nombres de las enfermedades a nuestro DataFrame. A diferencia de las variables anteriores que se mapearon con `replace()`, en este caso, es más eficiente usar una función de combinación de datos como `merge()`. La función `merge()` combina dos DataFrames basándose en valores comunes en una o más columnas, de manera similar a como se unen tablas en bases de datos. Veremos esta función en detalle más adelante.
Nota como se quitaron las columnas adicionales y se cambio el nombre de la columna. Como ejercicio, comenta y descomenta cada una de las lineas y ejecuta el notebook varias veces para ver que sucede.

> Nota: la función `.merge()` es muy poderosa y la veremos más adelante con detenimiento, puedes revisar su [documentación](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html).

In [11]:
# Realizar el merge para añadir la columna descrip
df_covid = pd.merge(
                    df_covid, # tabla a unir a la izquierda
                    mapeo_causa, # tabla a unir a la derecha
                    left_on='causa_def', # columna que se usara para unir en la tabla izquierda
                    right_on='cve', # columna que se usara para unir en la tabla izquierda
                    how='left' # forma de union, usaremos la tabla izquierda de base
                  )
# Quitar columna cve
df_covid = df_covid.drop(columns=['cve'])
# Renombrar columna descrip
df_covid = df_covid.rename(columns={'descrip':'causa_def_nom'})
#Revisar
df_covid.tail()

,dia_ocurr,mes_ocurr,anio_ocur,ent_ocurr,mun_ocurr,causa_def,sexo,edad,edad_agru,causa_def_nom
200258,31.0,12.0,2020,32,56,U071,Mujer,4057,De 55 a 59,"COVID-19, virus identificado"
200259,31.0,12.0,2020,32,56,U072,Mujer,4068,De 65 a 69,"COVID-19, virus no identificado"
200260,11.0,12.0,2020,32,17,U071,Hombre,4074,De 70 a 74,"COVID-19, virus identificado"
200261,18.0,12.0,2020,32,17,U071,Mujer,4081,De 80 a 84,"COVID-19, virus identificado"
200262,23.0,8.0,2020,32,56,U071,Mujer,4067,De 65 a 69,"COVID-19, virus identificado"


Para interpretar las variables de ubicación, `ent_ocurr` y `mun_ocurr`, es necesario convertirlas de códigos a nombres. La información para este mapeo se encuentra en la carpeta `catalogos` el archivo `entidad_municipio_localidad_2020.CSV`, que contiene los catálogos de entidades, municipios y localidades.

> Nota: es recomendable manejar este tipo de variables como `string`, ya que los ceros al principio de las claves son importantes en el catálogo del INEGI. Sin embargo, por simplicidad, los usamos como `int` en este ejemplo.

In [12]:
file_path = folder_path+'conjunto_de_datos_defunciones_registradas_2020_csv/catalogos/entidad_municipio_localidad_2020.CSV'
mapeo_lugar = pd.read_csv(file_path)
mapeo_lugar.head()

,cve_ent,cve_mun,cve_loc,nom_loc
0,1,0,0,Aguascalientes
1,1,1,0,Aguascalientes
2,1,1,1,Aguascalientes
3,1,1,106,Arellano
4,1,1,120,Buenavista de Peñuelas


La primera tarea es cargar el archivo del catálogo y prepararlo para el mapeo. Es importante notar que el catálogo incluye códigos especiales (`cve_mun` con `000` o `cve_loc` con `0000`) que representan el total de una entidad o un municipio.

Además. el catálogo es **jerárquico**. Esto significa que las claves del municipio (`cve_mun`) dependen de la clave de la entidad (`cve_ent`). Por lo tanto, no se puede mapear un municipio solo con su código, ya que podría estar repetido en diferentes entidades. Para lograr esto usaremos de nuevo la función `.merge()`.

Primero, mapearemos las entidades, estas se encuentran en la columna `ent_ocurr`.

In [13]:
# Seleccionar solo las entidades y columnas de interes
mapeo_ent = mapeo_lugar.loc[mapeo_lugar['cve_mun']==0, ['cve_ent','nom_loc']]
# cambiar nombre columnas
mapeo_ent.columns = ['ent_ocurr', 'ent_ocurr_nom']
display(mapeo_ent)

,ent_ocurr,ent_ocurr_nom
0,1,Aguascalientes
189,2,Baja California
372,3,Baja California Sur
443,4,Campeche
662,5,Coahuila de Zaragoza
973,6,Colima
1089,7,Chiapas
3453,8,Chihuahua
4059,9,Ciudad de México
4131,10,Durango


In [14]:
# Unir ambas tablas
df_covid = pd.merge(df_covid, mapeo_ent, on='ent_ocurr', how='left' )
#Revisar
df_covid[['ent_ocurr', 'ent_ocurr_nom']].tail()

,ent_ocurr,ent_ocurr_nom
200258,32,Zacatecas
200259,32,Zacatecas
200260,32,Zacatecas
200261,32,Zacatecas
200262,32,Zacatecas


Ahora seleccionaremos los nombres de municipios, es decir, aquellas filas del catálogo de lugares donde `cve_loc` es  `'0000'`. A continuación uniremos las dos tablas con `.merge()`. Cómo el catálogo es jerárquico es necesario tomar en cuenta dos columnas: `cve_ent` y `cve_mun`

In [15]:
# Seleccionar solo las entidades y columnas de interes
mapeo_mun = mapeo_lugar.loc[mapeo_lugar['cve_loc']==0, ['cve_ent', 'cve_mun','nom_loc']]
# cambiar nombre columnas
mapeo_mun.columns = ['ent_ocurr', 'mun_ocurr', 'mun_ocurr_nom']
mapeo_mun


,ent_ocurr,mun_ocurr,mun_ocurr_nom
0,1,0,Aguascalientes
1,1,1,Aguascalientes
27,1,2,Asientos
57,1,3,Calvillo
81,1,4,Cosío
...,...,...,...
30636,35,999,Municipio no especificado
30638,88,0,Entidad no aplica para A00 - R99 Y V90 - Y89
30639,88,888,Municipio no aplica para A00 - R99 Y V90 - Y89
30641,99,0,Entidad no especificada


In [16]:
# Unir ambas tablas
df_covid = pd.merge(df_covid, mapeo_mun, on=['ent_ocurr', 'mun_ocurr'], how='left' )
#Revisar
df_covid[['ent_ocurr', 'ent_ocurr_nom', 'mun_ocurr', 'mun_ocurr_nom']].tail()

,ent_ocurr,ent_ocurr_nom,mun_ocurr,mun_ocurr_nom
200258,32,Zacatecas,56,Zacatecas
200259,32,Zacatecas,56,Zacatecas
200260,32,Zacatecas,17,Guadalupe
200261,32,Zacatecas,17,Guadalupe
200262,32,Zacatecas,56,Zacatecas


## Manejo de columnas especiales



### Aplicando una función definida con `.apply()`

---



La variable `edad` utiliza un código numérico para representar la edad en diferentes escalas (días, meses, años). Para nuestro análisis, nos interesa la edad en años. Convertiremos los códigos de edades menores a un año a `0` y los códigos de años no especificados a `NaN` para facilitar los cálculos numéricos.

Para aplicar una lógica de conversión compleja a cada fila de una columna, crearemos una función especial `modify_age_inegi()` y la aplicaremos a la columna `edad` usando la función `.apply()`.

La función `.apply()` es una forma flexible de aplicar una función a cada elemento de una serie o a lo largo de los ejes de un DataFrame. Su sintaxis básica es `df[columna].apply(nombre_de_la_funcion)`. Es útil para operaciones personalizadas que no están disponibles en las funciones predefinidas de pandas.

El generar funciones especiales es una manera muy poderosa de hacer el tratamiento de datos en Python. A continuación, la función `modify_age_inegi` convierte los códigos de edad del INEGI:

  * Si el código es `4998`, que representa la edad no especificada, devuelve un valor nulo (`np.nan`).
  * Si el código es menor a `4000`, que corresponde a edades en días o meses, devuelve `0`.
  * Para los códigos que comienzan con `40` (edades en años), resta `4000` para obtener la edad real.

In [20]:
# Definimos la función especial
def modify_age_inegi(number):
    if number == 4998:
        return np.nan
    elif number < 4000:
        return 0
    else:
        return number - 4000

# Aplicar la función a la columna 'EDAD' y crear una nueva columna 'EDAD_ANOS'
df_covid['edad'] = df_covid['edad'].apply(modify_age_inegi)

# Mostrar los valores únicos de la nueva columna para verificar el cambio
df_covid['edad'].value_counts(dropna=False).sort_index()

,count
edad,
0.0,209
1.0,54
2.0,24
3.0,20
4.0,13
...,...
110.0,2
114.0,1
117.0,1


### Generar una columna `datetime`

La fecha se encuentra en tres columnas separadas. La biblioteca de python tiene un tipo `datetime` especializado para tiempo (fecha y hora), la cual tiene funciones especiales. Para poder usar estas funciones crearemos una columna de `fecha_ocur` con la función `pd.to_datetime()`.

In [21]:
# Obtener columnas de tiempo y renombrar
data_time = df_covid[['dia_ocurr', 'mes_ocurr', 'anio_ocur']] \
                .rename(columns={'dia_ocurr':'day', 'mes_ocurr':'month', 'anio_ocur':'year'})
# Generar columna datetime
df_covid['fecha_ocur'] = pd.to_datetime(data_time)
df_covid[['dia_ocurr', 'mes_ocurr', 'anio_ocur', 'fecha_ocur']].tail()

,dia_ocurr,mes_ocurr,anio_ocur,fecha_ocur
200258,31.0,12.0,2020,2020-12-31
200259,31.0,12.0,2020,2020-12-31
200260,11.0,12.0,2020,2020-12-11
200261,18.0,12.0,2020,2020-12-18
200262,23.0,8.0,2020,2020-08-23


La ventaja de crear una columna de tipo `datetime` es que hay una serie de funciones especiales para ese tipo de dato las cuales se pueden consultar en la [documentación de datetime](https://pandas.pydata.org/pandas-docs/stable/reference/arrays.html#api-arrays-datetime). Por ejemplo, podemos obtener el día de la semana con `.dt.day_name()`.

In [22]:
df_covid['fecha_ocur'].dt.day_name().value_counts()

,count
fecha_ocur,
Monday,29868
Tuesday,29032
Wednesday,28931
Thursday,28652
Sunday,28493
Saturday,27866
Friday,27391


## Revisión y conversión de tipos de datos

Antes de terminar es importante revisar que las columnas tengan el tipo de datos correcto con `.info()`.


In [23]:
df_covid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200263 entries, 0 to 200262
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   dia_ocurr      200233 non-null  float64       
 1   mes_ocurr      200240 non-null  float64       
 2   anio_ocur      200263 non-null  int64         
 3   ent_ocurr      200263 non-null  int64         
 4   mun_ocurr      200263 non-null  int64         
 5   causa_def      200263 non-null  object        
 6   sexo           200263 non-null  category      
 7   edad           200195 non-null  float64       
 8   edad_agru      200263 non-null  category      
 9   causa_def_nom  200263 non-null  object        
 10  ent_ocurr_nom  200263 non-null  object        
 11  mun_ocurr_nom  200263 non-null  object        
 12  fecha_ocur     200233 non-null  datetime64[ns]
dtypes: category(2), datetime64[ns](1), float64(3), int64(3), object(4)
memory usage: 17.2+ MB


El reordenamiento de los datos al final de la limpieza es importante para la consistencia y la legibilidad. Aunque no afecta los cálculos, al reorganizar las columnas en un orden lógico facilita la visualización y el trabajo.

En las operaciones anteriores las nuevas columnas se agregaron por defecto al final del DataFrame. Asi que ahora es necesario reordenar las columnas, para agruparlas temáticamente.

Además, vamos a restablecer el índice para que la numeración de las filas sea consecutiva, lo que facilita el acceso a los datos. Aunque el DataFrame original conservaba un orden implícito basado en el INEGI, los filtros de limpieza crearon discontinuidades en el índice que ahora se corrigen.

In [24]:
# Reordenar las columnas del DataFrame
df_covid = df_covid[[
                     'fecha_ocur', 'dia_ocurr', 'mes_ocurr', 'anio_ocur',
                     'ent_ocurr', 'ent_ocurr_nom', 'mun_ocurr', 'mun_ocurr_nom',
                     'causa_def', 'causa_def_nom', 'sexo', 'edad', 'edad_agru'
                   ]]
# Reiniciar el índice para que sea consecutivo y eliminar el índice original
df_covid = df_covid.reset_index(drop=True)
# Verificar
df_covid

,fecha_ocur,dia_ocurr,mes_ocurr,anio_ocur,ent_ocurr,ent_ocurr_nom,mun_ocurr,mun_ocurr_nom,causa_def,causa_def_nom,sexo,edad,edad_agru
0,2020-10-26,26.0,10.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,58.0,De 55 a 59
1,2020-10-26,26.0,10.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,72.0,De 70 a 74
2,2020-04-28,28.0,4.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,64.0,De 60 a 64
3,2020-05-13,13.0,5.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,46.0,De 45 a 49
4,2020-05-17,17.0,5.0,2020,1,Aguascalientes,1,Aguascalientes,U071,"COVID-19, virus identificado",Hombre,45.0,De 45 a 49
...,...,...,...,...,...,...,...,...,...,...,...,...,...
200258,2020-12-31,31.0,12.0,2020,32,Zacatecas,56,Zacatecas,U071,"COVID-19, virus identificado",Mujer,57.0,De 55 a 59
200259,2020-12-31,31.0,12.0,2020,32,Zacatecas,56,Zacatecas,U072,"COVID-19, virus no identificado",Mujer,68.0,De 65 a 69
200260,2020-12-11,11.0,12.0,2020,32,Zacatecas,17,Guadalupe,U071,"COVID-19, virus identificado",Hombre,74.0,De 70 a 74
200261,2020-12-18,18.0,12.0,2020,32,Zacatecas,17,Guadalupe,U071,"COVID-19, virus identificado",Mujer,81.0,De 80 a 84


## Guardar el conjunto de datos limpio

Una vez que hemos completado la limpieza y preparación de los datos, el paso final es guardar el DataFrame resultante. Almacenar la versión limpia del archivo evita tener que repetir todo el proceso de limpieza cada vez que se desee realizar un nuevo análisis.

Para esto, usaremos dos formatos comunes: `CSV` y `parquet`.

Un archivo `CSV` (Comma-Separated Values) es un formato de texto simple que almacena datos tabulares. Cada fila es un registro de datos y cada campo de un registro está separado por una coma. Tiene la ventaja de que es universal y legible. Puedes abrirlo con cualquier editor de texto o software de hojas de cálculo. Esto facilita la colaboración, ya que cualquier persona puede usarlo sin necesidad de software o librerías específicas. Sin embargo, no conserva el tipo de dato de las columnas ni la estructura del índice del DataFrame. Al volver a cargarlo, Pandas debe inferir los tipos de datos, lo que puede causar errores si la inferencia no es correcta.

El formato `parquet` almacena la tabla tal como está, incluyendo sus tipos de datos, índice y estructura. Su principal ventaja es que es eficiente y mantiene los tipos de datos de las columnas. Al cargar un archivo parquet, se obtiene un objeto idéntico al que se guardó. Esto es ideal para flujos de trabajo donde se requiere cargar y guardar datos sin perder la estructura ni el tipo de dato. Sin embargo, no es legible ni portable, solo se puede leer con la programas especializados.

Al trabajar con Google Colab es importante fijarse donde guardamos los archivos. En este caso incluiremos el `path` a la carpeta donde esta el proyecto para asegurar el orden de los archivos al guardar.

In [25]:
# Guardar el DataFrame en formato CSV
df_covid.to_csv(folder_path+'defunciones_covid19_2020.csv', index=False)
# Guardar el DataFrame en formato parquet
df_covid.to_parquet(folder_path+'defunciones_covid19_2020.parquet')

## Ejercicio

Expande la pregunta para incluir al menos dos variables nuevas. Realiza la limpieza de datos incluyendo esas dos variables. Guarda el conjunto de datos con un nombre nuevo, ya que se volverá a usar.

## Resumen

En esta lección hemos aprendido varios conceptos:

* Antes de empezar a limpiar los datos es necesario plantear una estrategia que incluya para cada variable/columna
* Se pueden quitar filas y columnas con `.drop()`
* Es necesario revisar los datos faltantes
* Para los datos categóricos
    * Hacer un mapeo de equivalencias con `.replace()` o `.merge()`
    * Convertir a tipo categórico con `.astype('category')`
    * Ordenar el catálogo si es necesario
* Se pueden generar funciones especiales y apicarlas con `.apply()`
* Se puede convertir a formato de fecha con `pd.to_datetime()`
* Es necesario validar la limpieza antes de guardar
    * En formato `CSV` para compartir  con `.to_csv()`
    * En formato `parquet` para analizar en Python con `.to_parquet()`



**¡Gracias!**